# 09 - Síntese da observabilidade SWOT

Este notebook consolida os resultados dos notebooks 01 a 08 e produz uma análise espacial adicional dos pixels PIXC da passagem controlada `cycle 055 pass 227 tile 127L`, sem baixar novos dados e sem realizar nova consulta Earthdata.

## Fluxo metodológico resumido

1. O notebook 01 confirmou cobertura potencial por metadados no envelope dos exutórios.
2. O notebook 02 selecionou candidatos SWOT leves e um PIXC de referência.
3. Os notebooks 03 e 05 testaram RiverSP, mas as feições ficaram longe dos exutórios.
4. O notebook 06 testou LakeSP, também sem suporte direto.
5. O notebook 07 mostrou pixels PIXC muito próximos dos pontos.
6. O notebook 08 interpretou as flags e indicou que os pixels mais próximos eram `land` ou `land_near_water`, não água.

## Conceitos usados

Cobertura potencial significa que o envelope espacial do produto intercepta a área. Proximidade PIXC significa que existem pixels próximos aos exutórios. Classe do pixel mais próximo indica se aquele pixel é terra, terra próxima à água ou água. A distribuição espacial de água avalia os pixels classificados como água no entorno dos pontos, não apenas o pixel mais próximo.

In [ ]:
from __future__ import annotations

import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box


In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'src' / 'check_environment.py').exists():
            return candidate
    fallback = Path.home() / 'mystorage' / 'PPGGAG1889' / 'atividade3_swot'
    if fallback.exists():
        return fallback
    raise RuntimeError('FALHA: rode este notebook dentro do repositorio atividade3_swot.')

PROJECT_ROOT = find_project_root(Path.cwd())
TABELAS = PROJECT_ROOT / 'outputs' / 'tabelas'
FIGURAS = PROJECT_ROOT / 'outputs' / 'figuras'
LOGS = PROJECT_ROOT / 'outputs' / 'logs'
PIXC_RAW_DIR = PROJECT_ROOT / 'dados' / 'raw' / 'swot' / 'pixc'

EXUTORIOS_CSV = PROJECT_ROOT / 'dados' / 'exutorios.csv'
OBS_CSV = TABELAS / 'observabilidade_exutorios.csv'
RIVERSP_255_CSV = TABELAS / 'validacao_riversp_exutorios.csv'
RIVERSP_227_CSV = TABELAS / 'validacao_riversp_top1_ranking_exutorios.csv'
LAKESP_CSV = TABELAS / 'validacao_lakesp_exutorios.csv'
PIXC_VALID_CSV = TABELAS / 'validacao_pixc_controlado_exutorios.csv'
PIXC_INTERP_CSV = TABELAS / 'validacao_pixc_interpretada_exutorios.csv'
PIXC_INTERP_SUMMARY_CSV = TABELAS / 'resumo_validacao_pixc_interpretada.csv'
PIXC_SUMMARY_CSV = TABELAS / 'resumo_pixc_controlado.csv'

OUT_SYNTHESIS = TABELAS / 'sintese_observabilidade_swot_exutorios.csv'
OUT_SUMMARY = TABELAS / 'resumo_sintese_observabilidade_swot.csv'
OUT_PIXELS = TABELAS / 'pixels_pixc_classificados_entorno_exutorios.csv'
OUT_PIXELS_SUMMARY = TABELAS / 'resumo_pixels_pixc_por_janela.csv'
OUT_HEATMAP = FIGURAS / '09_mapa_calor_agua_pixc_exutorios.png'
OUT_SYNTHESIS_FIG = FIGURAS / '09_sintese_observabilidade_swot.png'
LOG_FILE = LOGS / '09_sintese_observabilidade_swot.log'

for path in [TABELAS, FIGURAS, LOGS]:
    path.mkdir(parents=True, exist_ok=True)
logging.basicConfig(filename=LOG_FILE, filemode='w', level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
print('OK raiz:', PROJECT_ROOT)
print('OK log:', LOG_FILE)


## Leitura dos outputs anteriores

Arquivos essenciais ausentes geram erro explícito. Arquivos auxiliares são lidos quando existirem e marcados como ausentes quando não existirem.

In [ ]:
def require_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'FALHA: arquivo essencial ausente: {path}')
    return pd.read_csv(path)

def optional_csv(path: Path) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path)
    logging.warning('Arquivo opcional ausente: %s', path)
    return pd.DataFrame()

exutorios = require_csv(EXUTORIOS_CSV)
observabilidade = optional_csv(OBS_CSV)
riversp_255 = optional_csv(RIVERSP_255_CSV)
riversp_227 = optional_csv(RIVERSP_227_CSV)
lakesp = optional_csv(LAKESP_CSV)
pixc_valid = require_csv(PIXC_VALID_CSV)
pixc_interp = require_csv(PIXC_INTERP_CSV)
pixc_interp_summary = optional_csv(PIXC_INTERP_SUMMARY_CSV)
pixc_summary = require_csv(PIXC_SUMMARY_CSV)

expected_columns = ['id', 'latitude', 'longitude']
if list(exutorios.columns) != expected_columns or len(exutorios) != 13:
    raise ValueError(f'FALHA: exutorios deve ter 13 linhas e colunas {expected_columns}. Encontrado {len(exutorios)} linhas e {list(exutorios.columns)}')
print('OK arquivos lidos')
print('exutorios:', len(exutorios))
print('pixc interpretado:', len(pixc_interp))


## Tabela consolidada por exutório

A conclusão é conservadora: `suporte_agua_pixc` só é usado quando o pixel associado ao exutório pertence a uma classe de água. Pixels próximos classificados como terra ou terra próxima à água resultam em `sem_suporte_agua_na_passagem_testada`.

In [ ]:
def support_map(df: pd.DataFrame, support_col: str) -> dict:
    if df.empty or 'id' not in df.columns or support_col not in df.columns:
        return {}
    return df.set_index('id')[support_col].to_dict()

coverage = {}
if not observabilidade.empty and 'id' in observabilidade.columns and 'cobertura_potencial' in observabilidade.columns:
    coverage = observabilidade.set_index('id')['cobertura_potencial'].to_dict()

synthesis = exutorios.copy()
synthesis['cobertura_potencial_swot'] = synthesis['id'].map(coverage).fillna('indeterminado')
synthesis['suporte_riversp_pass255'] = synthesis['id'].map(support_map(riversp_255, 'suporte_riversp')).fillna('indeterminado')
synthesis['suporte_riversp_pass227'] = synthesis['id'].map(support_map(riversp_227, 'suporte_riversp')).fillna('indeterminado')
synthesis['suporte_lakesp_pass227'] = synthesis['id'].map(support_map(lakesp, 'suporte_lakesp')).fillna('indeterminado')

pix_cols = ['id', 'distancia_m_pixel_pixc', 'classe_pixel', 'classe_pixel_significado', 'suporte_pixc_interpretado']
missing_pix = [c for c in pix_cols if c not in pixc_interp.columns]
if missing_pix:
    raise ValueError(f'FALHA: colunas ausentes na validacao PIXC interpretada: {missing_pix}')
synthesis = synthesis.merge(pixc_interp[pix_cols], on='id', how='left')
synthesis = synthesis.rename(columns={'distancia_m_pixel_pixc': 'distancia_pixc_m'})

water_meanings = {'water_near_land', 'open_water', 'dark_water', 'low_coh_water_near_land', 'open_low_coh_water'}
def conclusion(row):
    if pd.isna(row.get('distancia_pixc_m')) or pd.isna(row.get('classe_pixel')):
        return 'indeterminado'
    if float(row['distancia_pixc_m']) > 500:
        return 'indeterminado'
    meaning = str(row.get('classe_pixel_significado', '')).strip()
    if meaning in water_meanings:
        return 'suporte_agua_pixc'
    if meaning in {'land', 'land_near_water'}:
        return 'sem_suporte_agua_na_passagem_testada'
    return 'indeterminado'

synthesis['conclusao_observabilidade'] = synthesis.apply(conclusion, axis=1)
ordered_cols = ['id','latitude','longitude','cobertura_potencial_swot','suporte_riversp_pass255','suporte_riversp_pass227','suporte_lakesp_pass227','distancia_pixc_m','classe_pixel','classe_pixel_significado','suporte_pixc_interpretado','conclusao_observabilidade']
synthesis = synthesis[ordered_cols]
synthesis.to_csv(OUT_SYNTHESIS, index=False, encoding='utf-8')
logging.info('Sintese por exutorio salva: %s', OUT_SYNTHESIS)
print('OK sintese salva:', OUT_SYNTHESIS)
display(synthesis)


In [ ]:
summary = pd.DataFrame([{
    'n_exutorios': len(synthesis),
    'n_cobertura_potencial': int(synthesis['cobertura_potencial_swot'].astype(str).str.lower().eq('sim').sum()),
    'n_suporte_riversp': int((synthesis['suporte_riversp_pass255'].astype(str).str.lower().eq('sim') | synthesis['suporte_riversp_pass227'].astype(str).str.lower().eq('sim')).sum()),
    'n_suporte_lakesp': int(synthesis['suporte_lakesp_pass227'].astype(str).str.lower().eq('sim').sum()),
    'n_pixel_pixc_proximo': int(pd.to_numeric(synthesis['distancia_pixc_m'], errors='coerce').le(500).sum()),
    'n_classe_pixc_agua_pixel_mais_proximo': int(synthesis['classe_pixel_significado'].isin(water_meanings).sum()),
    'n_sem_suporte_agua_na_passagem_testada': int(synthesis['conclusao_observabilidade'].eq('sem_suporte_agua_na_passagem_testada').sum()),
    'n_suporte_agua_pixc': int(synthesis['conclusao_observabilidade'].eq('suporte_agua_pixc').sum()),
    'n_indeterminado': int(synthesis['conclusao_observabilidade'].eq('indeterminado').sum()),
}])
summary.to_csv(OUT_SUMMARY, index=False, encoding='utf-8')
logging.info('Resumo sintese salvo: %s', OUT_SUMMARY)
print('OK resumo salvo:', OUT_SUMMARY)
display(summary)


## Interpretação das classes PIXC

A classificação dos pixels usa o dicionário obtido no notebook 08: classe `1 = terra`, classe `2 = terra_proxima_agua`, e classes `3`, `4`, `5`, `6`, `7 = agua`. Valores ausentes ou fora desse conjunto são `desconhecido`.

In [ ]:
try:
    import netCDF4
except Exception as exc:
    logging.exception('Falha ao importar netCDF4')
    raise RuntimeError('FALHA: netCDF4 nao esta instalado no ambiente.') from exc

granule_id = str(pixc_summary.iloc[0]['granule_id'])
nc_candidates = sorted(PIXC_RAW_DIR.glob(f'{granule_id}*.nc')) + sorted(PIXC_RAW_DIR.glob(f'{granule_id}*.nc4'))
if not nc_candidates:
    nc_candidates = sorted(PIXC_RAW_DIR.glob('*.nc')) + sorted(PIXC_RAW_DIR.glob('*.nc4'))
if not nc_candidates:
    raise FileNotFoundError(f'FALHA: arquivo PIXC .nc usado no notebook 07 nao encontrado em {PIXC_RAW_DIR}.')
nc_path = nc_candidates[0]
root_nc = netCDF4.Dataset(nc_path, mode='r')
print('OK PIXC:', nc_path)
print('granule_id:', granule_id)


In [ ]:
def walk_groups(group, path=''):
    yield path or '/', group
    for name, child in group.groups.items():
        yield from walk_groups(child, f'{path}/{name}' if path else f'/{name}')

def find_var(names: list[str]):
    wanted = {n.lower() for n in names}
    matches = []
    for group_path, group in walk_groups(root_nc):
        for name in group.variables.keys():
            if name.lower() in wanted:
                matches.append((group_path, name))
    for pref in ['/pixel_cloud', '/']:
        for match in matches:
            if match[0] == pref:
                return match
    return matches[0] if matches else (None, None)

def get_group(path: str):
    if path in ['', '/']:
        return root_nc
    group = root_nc
    for part in path.strip('/').split('/'):
        group = group.groups[part]
    return group

lat_group, lat_name = find_var(['latitude', 'lat'])
lon_group, lon_name = find_var(['longitude', 'lon'])
class_group, class_name = find_var(['classification', 'pixel_classification', 'class'])
quality_group, quality_name = find_var(['geolocation_qual', 'sig0_qual', 'classification_qual', 'quality_flag', 'qual', 'pixc_qual'])
if None in [lat_group, lon_group, class_group]:
    raise RuntimeError('FALHA: latitude, longitude ou classificacao nao encontradas no PIXC.')
lat = np.array(get_group(lat_group).variables[lat_name][:]).ravel()
lon = np.array(get_group(lon_group).variables[lon_name][:]).ravel()
classe = np.array(get_group(class_group).variables[class_name][:]).ravel()
qualidade = np.array(get_group(quality_group).variables[quality_name][:]).ravel() if quality_group else np.full(len(lat), np.nan)
valid_mask = np.isfinite(lat) & np.isfinite(lon) & (lat >= -90) & (lat <= 90) & (lon >= -180) & (lon <= 180)
print('pixels totais:', len(lat))
print('pixels validos:', int(valid_mask.sum()))
print('classe:', class_group, class_name)
print('qualidade:', quality_group, quality_name)


## Classificação dos pixels no entorno

Para lidar com milhões de pixels sem plotagem direta excessiva, o notebook recorta/classifica apenas pixels dentro de até 10 km dos exutórios e usa agregação em grade para o mapa de calor de água. Esse método é registrado no log.

In [ ]:
METRIC_CRS = 'EPSG:32723'
points = gpd.GeoDataFrame(exutorios.copy(), geometry=gpd.points_from_xy(exutorios['longitude'], exutorios['latitude']), crs='EPSG:4326')
points_m = points.to_crs(METRIC_CRS)

pix = pd.DataFrame({'latitude': lat[valid_mask], 'longitude': lon[valid_mask], 'classe_pixel': classe[valid_mask], 'qualidade_pixel': qualidade[valid_mask]})
pix_gdf = gpd.GeoDataFrame(pix, geometry=gpd.points_from_xy(pix['longitude'], pix['latitude']), crs='EPSG:4326')
pix_m = pix_gdf.to_crs(METRIC_CRS)
union_10km = points_m.buffer(10000).unary_union
near = pix_m[pix_m.geometry.within(union_10km)].copy()
logging.info('Pixels PIXC em ate 10 km: %s', len(near))

class_meaning = {1:'land', 2:'land_near_water', 3:'water_near_land', 4:'open_water', 5:'dark_water', 6:'low_coh_water_near_land', 7:'open_low_coh_water'}
def cat(code):
    try:
        c = int(code)
    except Exception:
        return 'desconhecido'
    if c == 1:
        return 'terra'
    if c == 2:
        return 'terra_proxima_agua'
    if c in [3,4,5,6,7]:
        return 'agua'
    return 'desconhecido'
near['pixel_id'] = np.arange(len(near), dtype=int)
near['classe_pixel_significado'] = near['classe_pixel'].map(lambda x: class_meaning.get(int(x), '') if pd.notna(x) else '')
near['categoria_agua'] = near['classe_pixel'].map(cat)

point_union = points_m.geometry.unary_union
near['distancia_min_exutorio_m'] = near.geometry.distance(point_union)
near['janela_2km'] = near['distancia_min_exutorio_m'].le(2000)
near['janela_5km'] = near['distancia_min_exutorio_m'].le(5000)
near['janela_10km'] = near['distancia_min_exutorio_m'].le(10000)

out_pixels = near.to_crs('EPSG:4326').copy()
out_pixels['latitude'] = out_pixels.geometry.y
out_pixels['longitude'] = out_pixels.geometry.x
out_pixels[['pixel_id','latitude','longitude','classe_pixel','classe_pixel_significado','categoria_agua','qualidade_pixel','distancia_min_exutorio_m','janela_2km','janela_5km','janela_10km']].to_csv(OUT_PIXELS, index=False, encoding='utf-8')
logging.info('Pixels classificados no entorno salvos: %s', OUT_PIXELS)
print('OK pixels classificados:', len(out_pixels))
display(out_pixels[['pixel_id','latitude','longitude','classe_pixel','classe_pixel_significado','categoria_agua','qualidade_pixel','distancia_min_exutorio_m','janela_2km','janela_5km','janela_10km']].head())


In [ ]:
rows = []
for window in [2000, 5000, 10000]:
    sub = near[near['distancia_min_exutorio_m'].le(window)]
    counts = sub['categoria_agua'].value_counts().to_dict()
    n = len(sub)
    rows.append({
        'janela_m': window,
        'n_pixels': int(n),
        'n_terra': int(counts.get('terra', 0)),
        'n_terra_proxima_agua': int(counts.get('terra_proxima_agua', 0)),
        'n_agua': int(counts.get('agua', 0)),
        'n_desconhecido': int(counts.get('desconhecido', 0)),
        'percentual_agua': round((counts.get('agua', 0) / n * 100), 3) if n else 0.0,
    })
pixel_summary = pd.DataFrame(rows)
pixel_summary.to_csv(OUT_PIXELS_SUMMARY, index=False, encoding='utf-8')
logging.info('Resumo por janela salvo: %s', OUT_PIXELS_SUMMARY)
display(pixel_summary)


## Mapa de calor de água

O mapa de calor usa somente pixels classificados como água dentro de 10 km. Os pixels são agregados em uma grade métrica de 250 m, reduzindo volume de plotagem e destacando concentração espacial de água.

In [ ]:
GRID_M = 250
water = near[near['categoria_agua'].eq('agua')].copy()
land_near = near[near['categoria_agua'].eq('terra_proxima_agua')].copy()
fig, ax = plt.subplots(figsize=(10, 9))
try:
    points_plot = points.to_crs('EPSG:4326')
    if not water.empty:
        wx = water.geometry.x.to_numpy(); wy = water.geometry.y.to_numpy()
        xmin, ymin, xmax, ymax = near.total_bounds
        xbins = np.arange(xmin, xmax + GRID_M, GRID_M)
        ybins = np.arange(ymin, ymax + GRID_M, GRID_M)
        hist, xedges, yedges = np.histogram2d(wx, wy, bins=[xbins, ybins])
        cells = []
        for ix in range(hist.shape[0]):
            for iy in range(hist.shape[1]):
                count = int(hist[ix, iy])
                if count > 0:
                    cells.append({'n_agua': count, 'geometry': box(xedges[ix], yedges[iy], xedges[ix+1], yedges[iy+1])})
        grid = gpd.GeoDataFrame(cells, crs=METRIC_CRS).to_crs('EPSG:4326')
        grid.plot(ax=ax, column='n_agua', cmap='Blues', alpha=0.75, legend=True, edgecolor='none', label='densidade de agua')
        logging.info('Mapa de calor agregado em grade de %s m com %s celulas.', GRID_M, len(grid))
    else:
        logging.warning('Nenhum pixel de agua encontrado em ate 10 km; mapa de calor sem celulas de agua.')
    if not land_near.empty:
        sample_land_near = land_near.sample(min(len(land_near), 5000), random_state=42).to_crs('EPSG:4326')
        sample_land_near.plot(ax=ax, color='#fdae6b', markersize=1.5, alpha=0.45, label='terra_proxima_agua')
    points_plot.plot(ax=ax, color='black', markersize=45, label='Exutorios', zorder=5)
    for _, row in points_plot.iterrows():
        ax.annotate(row['id'], (row.geometry.x, row.geometry.y), xytext=(4, 4), textcoords='offset points', fontsize=8)
    minx, miny, maxx, maxy = points_plot.total_bounds
    pad_x = max((maxx - minx) * 5, 0.06); pad_y = max((maxy - miny) * 5, 0.06)
    ax.set_xlim(minx - pad_x, maxx + pad_x); ax.set_ylim(miny - pad_y, maxy + pad_y)
    ax.set_title('Mapa de calor de agua PIXC - cycle 055 pass 227 tile 127L')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.25)
    fig.tight_layout(); fig.savefig(OUT_HEATMAP, dpi=180)
    print('OK mapa de calor salvo:', OUT_HEATMAP)
    plt.show()
except Exception as exc:
    logging.exception('Falha ao gerar mapa de calor PIXC')
    raise RuntimeError('FALHA: nao foi possivel gerar o mapa de calor PIXC.') from exc


## Figura final de síntese

A figura final mostra cada exutório colorido pela conclusão conservadora da observabilidade na passagem testada.

In [ ]:
points_syn = gpd.GeoDataFrame(synthesis.copy(), geometry=gpd.points_from_xy(synthesis['longitude'], synthesis['latitude']), crs='EPSG:4326')
colors = {'sem_suporte_agua_na_passagem_testada':'#cb181d', 'suporte_agua_pixc':'#238b45', 'indeterminado':'#f0ad00'}
fig, ax = plt.subplots(figsize=(9, 8))
for status, sub in points_syn.groupby('conclusao_observabilidade'):
    sub.plot(ax=ax, color=colors.get(status, '#737373'), markersize=70, label=status, edgecolor='black', linewidth=0.5)
for _, row in points_syn.iterrows():
    ax.annotate(row['id'], (row.geometry.x, row.geometry.y), xytext=(4, 4), textcoords='offset points', fontsize=8)
minx, miny, maxx, maxy = points_syn.total_bounds
pad_x = max((maxx - minx) * 2.5, 0.01); pad_y = max((maxy - miny) * 2.5, 0.01)
ax.set_xlim(minx - pad_x, maxx + pad_x); ax.set_ylim(miny - pad_y, maxy + pad_y)
ax.set_title('Sintese observabilidade SWOT/PIXC testada - cycle 055 pass 227')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.legend(loc='best')
ax.grid(True, alpha=0.25)
fig.tight_layout(); fig.savefig(OUT_SYNTHESIS_FIG, dpi=180)
logging.info('Figura sintese salva: %s', OUT_SYNTHESIS_FIG)
print('OK figura sintese salva:', OUT_SYNTHESIS_FIG)
plt.show()


## Principais resultados esperados

A síntese deve evidenciar que houve cobertura potencial e pixels PIXC próximos, mas que a classe do pixel mais próximo não indicou água para os 13 exutórios na passagem controlada. O mapa de calor ajuda a verificar se há água no entorno, mesmo quando o pixel exatamente associado ao ponto não é água.

## Limitações e próximos passos

A análise usa uma única passagem PIXC e um limiar operacional de 500 m para o pixel mais próximo. A grade de 250 m do mapa de calor é uma agregação visual, não uma medida hidrológica final. Como próximo passo, recomenda-se usar a distribuição espacial de água para escolher um recorte PIXC mais direcionado ou testar outra passagem, sempre documentando a justificativa espacial.